# Parameter Grid Search

Systematically test Suite2p/Cellpose parameter combinations to find optimal settings.

**Key Features:**
- When searching only detection parameters, registration runs once and is reused
- Automatic quality metrics collection (SNR, skewness, shot noise)
- Built-in comparison visualizations

In [ ]:
from pathlib import Path
import lbm_suite2p_python as lsp

# paths
data_dir = Path(r"D:/demo/raw")
save_path = Path(r"D:/demo/grid_search")

## Define Parameters

**Detection parameters** (fast - registration runs once):
- `threshold_scaling`, `diameter`, `spatial_hp_cp`, `anatomical_only`, `sparse_mode`

**Registration parameters** (slow - each combination re-registers):
- `nonrigid`, `block_size`, `maxregshift`, `smooth_sigma`

In [ ]:
# parameters to search
grid_params = {
    "spatial_hp_cp": [0, 0.5, 3, 10],
}

# fixed parameters
ops = {
    "anatomical_only": 4,
    "diameter": 4,
}

## Run Grid Search

In [ ]:
lsp.grid_search(
    input_data=data_dir,
    save_path=save_path,
    grid_params=grid_params,
    ops=ops,
    planes=7,
    force_detect=True,
)

## Collect and Analyze Results

In [ ]:
# collect metrics from all combinations
df = lsp.collect_grid_results(save_path, grid_params)
display(df[["combo", "n_accepted", "snr_median", "skew_median", "noise_median"] + list(grid_params.keys())])

# save to CSV
lsp.save_grid_results(df, save_path)

## Best Parameters

In [ ]:
lsp.print_best_parameters(df, grid_params)

## Visualize Quality Metrics

In [ ]:
lsp.plot_grid_metrics(df, grid_params, save_path=save_path / "quality_metrics.png")

## Distribution Comparison

In [ ]:
lsp.plot_grid_distributions(df, results_dir=save_path, n_top=4, save_path=save_path / "distributions.png")

## Compare Detection Masks

In [ ]:
lsp.plot_grid_masks(df, results_dir=save_path, n_top=4, save_path=save_path / "masks.png")

## Output Structure

```
grid_search/
├── _base/                    # shared registration (detection-only search)
│   ├── data_raw.bin
│   ├── data.bin
│   └── ops.npy
├── grid_search_results.csv   # all metrics
├── quality_metrics.png
├── distributions.png
├── masks.png
├── spa0/
│   ├── ops.npy, stat.npy, F.npy, ...
├── spa0.50/
└── ...
```

## Using Best Parameters

```python
best = lsp.get_best_parameters(df)
best_ops = {"spatial_hp_cp": best["best_snr"]["spatial_hp_cp"]}

lsp.pipeline(
    input_data="D:/data/full_volume",
    save_path="D:/results/full_run",
    ops=best_ops,
)
```